<a href="https://colab.research.google.com/github/Mansik-04/assignments/blob/main/detecron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1: Purpose of object tracking and difference from detection
# - Object detection locates and classifies objects in individual frames (bounding boxes, class labels).
# - Object tracking assigns persistent IDs to detected objects across frames, producing object trajectories over time. Tracking enables analytics like counting, path analysis, dwell time, and behavior understanding.
#
# Q2: Role of Kalman Filter
# - Kalman Filter is a recursive estimator used to predict an object's next state (position, velocity) and update that prediction with new noisy measurements (detections). It is commonly used because it's computationally efficient, handles motion prediction well, and integrates naturally with data association frameworks.
#
# Q3: Importance of annotation for tracking
# - High-quality frame-by-frame annotations with object IDs are required to train and evaluate tracking systems. Tools include CVAT, LabelMe, VATIC, and motchallenge-format converters. Annotation includes bounding boxes and consistent ID assignment across frames.
#
# Q4: Selecting a pre-trained model from model zoo
# - Steps: identify task (detection vs detection+reid), choose compatible model (e.g., YOLO, Faster R-CNN), check input size & pretraining dataset, download weights, adapt heads (num classes), fine-tune on custom dataset, and validate performance.
#
# Q5: Purpose of combining YOLO and Deep SORT
# - YOLO provides fast, accurate detections per frame. Deep SORT adds tracking by performing data association (Kalman filter + Hungarian algorithm) and re-identification using appearance features to maintain consistent IDs through occlusions and brief misses. This pipeline yields real-time multi-object tracking with fewer ID switches.

In [2]:
import sys

def iou(bbox1, bbox2):
    """
    Computes the Intersection over Union (IoU) of two bounding boxes.

    Args:
        bbox1: A tuple (x1, y1, x2, y2) representing the first bounding box.
        bbox2: A tuple (x1, y1, x2, y2) representing the second bounding box.

    Returns:
        The IoU value.
    """
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


# evaluation counters
FP = 0
FN = 0
ID_switches = 0
total_matches = 0
sum_iou = 0
num_matches = 0


# mapping from gt id to last matched track id to detect ID switches
gt_last_track = {}

# Assuming 'gt' and 'tracks' are lists of frames, where each frame
# is a list of objects with 'bbox' and 'id' (for gt) or 'track_id' (for tracks)
# Placeholder data - replace with your actual data
gt = [] # Example: [[{'bbox': (x1, y1, x2, y2), 'id': 1}, ...], ...]
tracks = [] # Example: [[{'bbox': (x1, y1, x2, y2), 'track_id': 1}, ...], ...]


for f_idx in range(min(len(gt), len(tracks))):
    gt_objs = gt[f_idx]
    tr_objs = tracks[f_idx]
    matched_gt = set()
    matched_tr = set()
    # compute IoU between every pair
    for i, g in enumerate(gt_objs):
        best_iou = 0
        best_j = -1
        for j, trk in enumerate(tr_objs):
            if j in matched_tr:
                continue
            val = iou(g['bbox'], trk['bbox'])
            if val > best_iou:
                best_iou = val
                best_j = j
        if best_j >= 0 and best_iou >= 0.5:
            # match
            matched_gt.add(i)
            matched_tr.add(best_j)
            total_matches += 1
            sum_iou += best_iou
            num_matches += 1
            gt_id = g['id']
            track_id = tr_objs[best_j]['track_id']
            if gt_id in gt_last_track and gt_last_track[gt_id] != track_id:
                ID_switches += 1
            gt_last_track[gt_id] = track_id
    # unmatched ground truth are false negatives
    FN += len(gt_objs) - len(matched_gt)
    # unmatched detections are false positives
    FP += len(tr_objs) - len(matched_tr)


MOTA = 1 - (FN + FP + ID_switches) / (sum(len(g) for g in gt) if sum(len(g) for g in gt) > 0 else 1)
MOTP = (sum_iou / num_matches) if num_matches > 0 else 0


print(f'Evaluation results on synthetic video:\nMOTA={MOTA:.3f}, MOTP={MOTP:.3f}, FN={FN}, FP={FP}, ID_switches={ID_switches}')

Evaluation results on synthetic video:
MOTA=1.000, MOTP=0.000, FN=0, FP=0, ID_switches=0
